# Hyperparameter Tuning with Optuna
# 使用 Optuna 进行超参数调优

This tutorial demonstrates how to use Optuna for hyperparameter optimization with PipelineTS.
本教程展示如何使用 Optuna 对 PipelineTS 进行超参数优化。

Contents:
内容：

1. **Basic Optuna integration / 基础 Optuna 集成**
2. **Tuning ML models / 调优 ML 模型**
3. **Tuning NN models / 调优 NN 模型**
4. **Analyzing results / 分析结果**

## Prerequisites / 前置条件

Install Optuna if not already installed:
如果尚未安装 Optuna，请先安装：

```bash
pip install optuna
```

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# Load data / 加载数据
from PipelineTS.dataset import LoadMessagesSentDataSets

init_data = LoadMessagesSentDataSets()
time_col = 'date'
target_col = 'ta'
init_data = init_data[[time_col, target_col]]
init_data[time_col] = pd.to_datetime(init_data[time_col])

# Split train and validation sets
# 划分训练集和验证集
n = 30
valid_data = init_data.iloc[-n:, :]
data = init_data.iloc[:-n, :]

print(f"Train shape / 训练集形状: {data.shape}")
print(f"Valid shape / 验证集形状: {valid_data.shape}")

## 1. Tuning ML Models with Optuna
## 1. 使用 Optuna 调优 ML 模型

Define an objective function that Optuna will minimize. The function creates a pipeline with suggested hyperparameters and returns the evaluation metric.
定义一个 Optuna 将最小化的目标函数。该函数使用建议的超参数创建管道并返回评估指标。

In [ ]:
import optuna
from PipelineTS.pipeline import ModelPipeline
from PipelineTS.ml_model import WideGBRTModel

def objective_ml(trial):
    # Suggest hyperparameters / 建议超参数
    lags = trial.suggest_int('lags', 8, 60, step=2)
    n_estimators = trial.suggest_int('n_estimators', 100, 500, log=True)
    differential_n = trial.suggest_int('differential_n', 0, 3)

    pipeline = ModelPipeline(
        time_col=time_col,
        target_col=target_col,
        lags=lags,
        random_state=42,
        include_models=WideGBRTModel,
        metric=mean_absolute_error,
        metric_less_is_better=True,
        scaler=None,
        WideGBRTModel__n_estimators=n_estimators,
        WideGBRTModel__differential_n=differential_n,
    )

    pipeline.fit(data, valid_data=valid_data)
    prediction = pipeline.predict(n)
    return mean_absolute_error(
        valid_data[target_col].values,
        prediction[target_col].values
    )

# Run optimization / 运行优化
study_ml = optuna.create_study(direction='minimize')
study_ml.optimize(objective_ml, n_trials=10)

print(f"\nBest parameters / 最佳参数: {study_ml.best_params}")
print(f"Best MAE / 最佳 MAE: {study_ml.best_value:.2f}")

## 2. Tuning NN Models
## 2. 调优 NN 模型

Neural network models have additional hyperparameters like learning rate, epochs, and architecture parameters.
神经网络模型有额外的超参数，如学习率、训练轮数和架构参数。

In [ ]:
from PipelineTS.nn_model import TCNModel

def objective_nn(trial):
    # Suggest hyperparameters / 建议超参数
    lags = trial.suggest_int('lags', 8, 48, step=4)
    epochs = trial.suggest_int('epochs', 100, 500, step=100)
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)

    model = TCNModel(
        time_col=time_col,
        target_col=target_col,
        lags=lags,
        epochs=epochs,
        learning_rate=learning_rate,
        kernel_size=kernel_size,
        patience=50,
        verbose=False,
        random_state=42,
    )
    model.fit(data)
    prediction = model.predict(n)
    return mean_absolute_error(
        valid_data[target_col].values,
        prediction[target_col].values
    )

# Run optimization / 运行优化
study_nn = optuna.create_study(direction='minimize')
study_nn.optimize(objective_nn, n_trials=5)

print(f"\nBest parameters / 最佳参数: {study_nn.best_params}")
print(f"Best MAE / 最佳 MAE: {study_nn.best_value:.2f}")

## 3. Tuning Multiple Models Simultaneously
## 3. 同时调优多个模型

You can also tune which model to use as a hyperparameter itself.
你也可以将模型选择本身作为超参数进行调优。

In [ ]:
def objective_multi(trial):
    # Suggest which model to use / 建议使用哪个模型
    model_name = trial.suggest_categorical('model', ['torch_boosting_forest', 'torch_bagging_forest', 'wide_gbrt'])
    lags = trial.suggest_int('lags', 10, 60, step=5)

    pipeline = ModelPipeline(
        time_col=time_col,
        target_col=target_col,
        lags=lags,
        random_state=42,
        include_models=[model_name],
        metric=mean_absolute_error,
        metric_less_is_better=True,
        scaler=None,
    )

    pipeline.fit(data, valid_data=valid_data)
    prediction = pipeline.predict(n)
    return mean_absolute_error(
        valid_data[target_col].values,
        prediction[target_col].values
    )

study_multi = optuna.create_study(direction='minimize')
study_multi.optimize(objective_multi, n_trials=10)

print(f"\nBest parameters / 最佳参数: {study_multi.best_params}")
print(f"Best MAE / 最佳 MAE: {study_multi.best_value:.2f}")

## 4. Analyzing Results
## 4. 分析结果

Optuna provides built-in visualization tools to analyze the optimization results.
Optuna 提供内置的可视化工具来分析优化结果。

In [ ]:
# Display all trials / 显示所有试验
trials_df = study_ml.trials_dataframe()
print("All trials (sorted by value) / 所有试验（按值排序）:")
trials_df.sort_values('value').head(10)

In [ ]:
# Train the final model with best parameters
# 使用最佳参数训练最终模型
best_params = study_ml.best_params
print(f"Training with best params / 使用最佳参数训练: {best_params}")

final_pipeline = ModelPipeline(
    time_col=time_col,
    target_col=target_col,
    lags=best_params['lags'],
    random_state=42,
    include_models=WideGBRTModel,
    metric=mean_absolute_error,
    metric_less_is_better=True,
    scaler=None,
    WideGBRTModel__n_estimators=best_params['n_estimators'],
    WideGBRTModel__differential_n=best_params['differential_n'],
)

final_pipeline.fit(data, valid_data=valid_data)
prediction = final_pipeline.predict(n)

from PipelineTS.plot import plot_data_period
plot_data_period(
    init_data.iloc[-100:, :], prediction,
    time_col=time_col, target_col=target_col,
    labels=['Ground Truth / 真实值', 'Prediction / 预测值']
)

## Summary / 总结

Key steps for hyperparameter tuning with Optuna:
使用 Optuna 进行超参数调优的关键步骤：

1. **Define objective function**: Create a function that takes a `trial` object, builds a model with suggested params, and returns the metric.
1. **定义目标函数**：创建一个接收 `trial` 对象的函数，使用建议的参数构建模型并返回指标。

2. **Create study**: `optuna.create_study(direction='minimize')` for metrics where lower is better.
2. **创建 study**：对于越低越好的指标使用 `optuna.create_study(direction='minimize')`。

3. **Run optimization**: `study.optimize(objective, n_trials=N)` to search for best parameters.
3. **运行优化**：`study.optimize(objective, n_trials=N)` 搜索最佳参数。

4. **Use best params**: `study.best_params` contains the optimal hyperparameters.
4. **使用最佳参数**：`study.best_params` 包含最优超参数。